# Protocolo de Análisis Estadístico Bivariante (`heavystats.bivariate`)

Este cuaderno interactivo implementa y documenta la metodología completa del **Análisis Estadístico Bivariante** para el biomonitoreo de metales pesados en sangre (**Plomo [Pb]**, **Mercurio [Hg]** y **Cadmio [Cd]**) en población infantil, conforme al protocolo maestro bioestadístico y toxicológico.

### Fundamentación Bioestadística y Control Metodológico
1. **Población Total vs Muestra Efectiva**: La caracterización general se realiza sobre la población total ($N=48$), mientras que cada análisis metal-factor se ejecuta estrictamente sobre los casos válidos disponibles para dicho biomarcador ($n=20$).
2. **Principio de Metal Individual**: A excepción de la co-exposición inter-metales (Etapa 11), todos los contrastes bivariantes se aplican a un metal por análisis para garantizar rigor inferencial.
3. **Enfoque No Paramétrico y Remuestreo Bootstrap**: Ante el tamaño muestral ($n=20$) y la asimetría de las concentraciones, se emplean Mediana [RIQ], estimador de Hodges-Lehmann, Mann–Whitney $U$ con correlación biserial por rangos ($r_{rb}$), Kruskal–Wallis $H$ con $\epsilon^2$, y Spearman $\rho_s$ con intervalos de confianza al 95% calculados por remuestreo Bootstrap (2,000 réplicas).
4. **Control de Multiplicidad**: Control de la Tasa de Falso Descubrimiento (FDR) mediante el procedimiento de **Benjamini-Hochberg** ($q < 0.10$) distinguiendo el valor $p$ crudo de $p_{FDR}$.
5. **Insumo para el Modelado Multivariante**: Ranking multicriterio de variables (Prioridad Alta, Intermedia y Baja), diagnóstico de colinealidad y preparación formal de bloques para PCA y PLS con restricción de parsimonia.

--- 
## 1. Auditoría, Limpieza y Clasificación de Variables

Aprovechamos los módulos de preprocesamiento, control de calidad (`heavystats.cleaning`, `heavystats.validation`) y descriptivos univariantes (`heavystats.univariate`) ya desarrollados.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import heavystats as hs
from heavystats.bivariate import BivariateTables, BivariatePlots
from heavystats.univariate import UnivariateTables, UnivariatePlots

# 1. Cargar datos brutos
df_raw = hs.load_data()

# 2. Control de Calidad y Validación
report_val = hs.validate_data(df_raw)

# 3. Pipeline de limpieza, estandarización y creación de indicadores compuestos
df_proc = hs.standardize_boolean_columns(df_raw)
df_proc = hs.desaggregate_multiple_responses(
    df_proc,
    columns=["Salud_Transporte", "Salud_Agua", "Exposicion_Talleres", "Exposicion_Lugares", "Exposicion_Industrias"]
)
df_proc = hs.encode_dietary_frequencies(df_proc)
df_proc = hs.create_composite_indicators(df_proc)

# 4. Extraer muestra analítica con biomarcadores séricos (n=20)
df_analytical = hs.get_analytical_sample(df_proc)
print(f"Población total de referencia: N = {df_proc.shape[0]} participantes.")
print(f"Muestra analítica cargada con éxito: n = {df_analytical.shape[0]} pacientes y {df_analytical.shape[1]} variables.")


Población total de referencia: N = 48 participantes.
Muestra analítica cargada con éxito: n = 20 pacientes y 61 variables.


In [ ]:
# 5. Clasificación Maestra de Variables (Etapa 1)
report_vars = hs.variables_table(df_analytical)
display(report_vars)

# 6. Perfil Descriptivo Univariante de Metales (Etapa 3)
u_tables = UnivariateTables(df_analytical, df_total=df_proc)
display(u_tables.metal_summary())

--- 
## 2. Inicialización de los Motores Bivariantes

Instanciamos las clases principales `BivariateTables` (para tablas científicas con estilo Booktabs y exportación Excel/CSV) y `BivariatePlots` (para visualizaciones a 300 DPI con estilo editorial y límites toxicológicos).

In [3]:
bt = BivariateTables(df_analytical)
bp = BivariatePlots(df_analytical)
print("Motores BivariateTables y BivariatePlots listos.")

Motores BivariateTables y BivariatePlots listos.


--- 
## Metal vs Variables Cuantitativas y Antropométricas

**Objetivo**: Cuantificar la asociación no lineal/monotónica entre las concentraciones de metales (**Plomo**, **Mercurio** y **Cadmio**) y variables numéricas continuas (**Edad**, **Peso**, **Altura**, **IMC**, **Score de Riesgo**).

Se calcula el coeficiente de correlación de Spearman ($\rho_s$) con intervalos de confianza al 95% obtenidos por remuestreo Bootstrap (2,000 réplicas).

In [4]:
# 4.1 Tabla: Correlaciones de Spearman para Plomo (Pb)
report_cont_pb = bt.continuous_summary(metal="Plomo_ug_dL")
display(report_cont_pb)

<BivariateTableReport shape=(5, 6) title='Asociación de Conc. de Plomo en Sangre (µg/dL) con Variables Cuantitativas'>

In [ ]:
# 4.2 Gráficos de dispersión continua con ajuste e intervalos de confianza al 95%
fig1, ax1 = bp.plot_continuous(metal="Plomo_ug_dL", column="Edad")
plt.show()

fig2, ax2 = bp.plot_continuous(metal="Plomo_ug_dL", column="Score_Riesgo")
plt.show()


--- 
## ETAPAS 5, 8 y 9: Metal vs Factores Binarios, Exposiciones, Hábitos y Síntomas

**Objetivo**: Evaluar si las concentraciones metálicas difieren significativamente según la presencia o ausencia de factores de riesgo, fuentes ambientales, hábitos y síntomas clínicos.

Se aplica la prueba no paramétrica de **Mann–Whitney $U$**, reportando Medianas [RIQ], el tamaño de efecto biserial por rangos ($r_{rb}$) con IC 95% Bootstrap y el estimador de localización de **Hodges-Lehmann**.

In [ ]:
# 5.1 Tabla: Comparación de Plomo según factores dicotómicos organizados por bloques temáticos
report_bin_pb = bt.binary_summary(metal="Plomo_ug_dL")
display(report_bin_pb)

In [ ]:
# 5.2 Boxplots con stripplot (puntos individuales), brackets estadísticos y límites permisibles
fig3, ax3 = bp.plot_binary(metal="Plomo_ug_dL", group_col="Es_Expuesto")
plt.show()

fig4, ax4 = bp.plot_binary(metal="Plomo_ug_dL", group_col="Exposicion_Lugares_estacion_gasolina")
plt.show()

# 5.3 Cuadrícula exploratoria integrada (Grid) para factores ambientales y de salud
fig_grid = bp.plot_grid(metal="Plomo_ug_dL", n_cols=3)
plt.show()


--- 
## ETAPA 6: Metal vs Variables Categóricas Politómicas

**Objetivo**: Evaluar diferencias de concentraciones a través de más de dos grupos nominales (**Sector residencial** e **Institución educativa**).

Se aplica la prueba de **Kruskal–Wallis $H$**, el tamaño de efecto **Epsilon Cuadrado ($\epsilon^2$)** y comparaciones múltiples post-hoc mediante la prueba de **Dunn** con ajuste de Holm-Bonferroni.

In [ ]:
# 6.1 Tabla: Comparación por Sector e Institución con Kruskal-Wallis
report_cat_pb = bt.categorical_summary(metal="Plomo_ug_dL", columns=["Sector", "Institucion"])
display(report_cat_pb)

# Desglose post-hoc de comparaciones por pares de Dunn
print("Comparaciones Post-Hoc de Dunn (Ajuste Holm):")
display(report_cat_pb.posthoc_df)

In [ ]:
# 6.2 Diagrama de caja por Sector residencial con anotación de Kruskal-Wallis
fig5 = bp.plot_categorical(metal="Plomo_ug_dL", group_col="Sector")
plt.show()

--- 
## ETAPA 7: Hábitos de Consumo Dietario (Variables Ordinales 0 a 4)

**Objetivo**: Evaluar la asociación entre la frecuencia de consumo alimenticio (0: Nunca a 4: Diario) y la concentración del metal, examinando especialmente hipótesis de plausibilidad biológica (ej. Consumo de Pescados $\to$ Mercurio).

Se utiliza correlación ordinal no paramétrica de Spearman ($\rho_s$) con IC 95% Bootstrap.

In [ ]:
# 7.1 Tabla: Hábitos dietarios vs Plomo y Mercurio
report_diet_pb = bt.dietary_summary(metal="Plomo_ug_dL")
display(report_diet_pb)

report_diet_hg = bt.dietary_summary(metal="Mercurio_ug_L")
display(report_diet_hg)

In [ ]:
# 7.2 Gráfico de tendencia ordinal de consumo de pescado vs Mercurio
fig6 = bp.plot_dietary(metal="Mercurio_ug_L", dietary_col="Alim_Pescados")
plt.show()

--- 
## ETAPA 10: Validación Empírica del Algoritmo de Riesgo

**Objetivo**: Determinar si un mayor `Score_Riesgo` o la presencia del indicador específico (`Riesgo_Pb`, `Riesgo_Hg`, `Riesgo_Cd`) se acompaña de concentraciones biológicas significativamente mayores.

In [ ]:
# 10.1 Calibración del Score de Riesgo frente a Plomo
report_risk_pb = bt.risk_score_summary(metal="Plomo_ug_dL")
display(report_risk_pb)

# 10.2 Dispersión con ajuste para Score de Riesgo vs Plomo
fig7, ax7 = bp.plot_continuous(metal="Plomo_ug_dL", column="Score_Riesgo")
plt.show()

# 10.3 Panel multi-metal de validación empírica frente a límites CDC/OMS/EPA
fig_risk_panel, _ = bp.plot_risk_algorithm()
plt.show()


--- 
## ETAPA 11: Co-Exposición Inter-Metálica (Metal ↔ Metal)

**Objetivo**: Analizar la co-ocurrencia y correlación sinérgica entre los tres biomarcadores principales (**Pb ↔ Hg**, **Pb ↔ Cd**, **Hg ↔ Cd**) para detectar fuentes comunes de contaminación ambiental o alimentaria.

In [ ]:
# 11.1 Tabla: Matriz de Co-Exposición Inter-Metales
report_metals_corr = bt.metal_correlations()
display(report_metals_corr)

In [ ]:
# 11.2 Heatmap de correlación de Spearman inter-metales con IC 95% Bootstrap
fig8, ax8 = bp.plot_metal_matrix()
plt.show()

# 11.3 Matriz de dispersión pareada (Pairplot) con KDE en diagonal y regresión
fig_pairplot, _ = bp.plot_metal_pairplot()
plt.show()


--- 
## ETAPAS 12, 13 y 14: Estratificación Toxicológica en Niveles (Bajo / Medio / Alto)

**Objetivo**: Clasificar los biomarcadores según límites de referencia internacionales (CDC, EPA, OMS) y evaluar el gradiente biológico del Score de Riesgo.

In [ ]:
# 12.1 Tabla: Clasificación de Plomo en 3 niveles toxicológicos
report_levels_pb = bt.metal_levels_summary(metal="Plomo_ug_dL")
display(report_levels_pb)

# 13.1 Tabla: Gradiente del Score de Riesgo a través de los niveles (Kruskal-Wallis y Jonckheere-Terpstra)
report_risk_levels = bt.risk_by_metal_level(metal="Plomo_ug_dL")
display(report_risk_levels)

--- 
## ETAPAS 17 y 18: Matriz Maestra de Screening Bivariante y Ranking de Variables

**Objetivo**: Ejecutar un tamizaje masivo sobre el conjunto exhaustivo de variables controlando la Tasa de Falso Descubrimiento (**Benjamini-Hochberg FDR**, $q < 0.10$) y generar un ranking multicriterio de predictores para alimentar PCA y PLS.

In [ ]:
# 17.1 Matriz Maestra de Asociaciones con ajuste FDR para Plomo
report_master_pb = bt.master_association_matrix(metal="Plomo_ug_dL", fdr_method="fdr_bh")
display(report_master_pb)

In [ ]:
# 17.2 Volcano Plot: Tamaño del efecto vs -log10(p) con umbral FDR
fig9 = bp.plot_volcano(metal="Plomo_ug_dL", fdr_alpha=0.10, p_alpha=0.05)
plt.show()

In [ ]:
# 18.1 Ranking Multicriterio de Variables (Prioridad Alta, Intermedia y Baja)
report_rank_pb = bt.variable_ranking(metal="Plomo_ug_dL")
display(report_rank_pb)

In [ ]:
# 18.2 Forest Plot de tamaños de efecto con intervalos de confianza al 95% Bootstrap
fig10 = bp.plot_forest_effects(metal="Plomo_ug_dL", top_n=15)
plt.show()

--- 
## ETAPAS 19, 20, 21 y 22: Colinealidad y Preparación para PCA / PLS

**Objetivo**: Diagnosticar colinealidad y dependencias estructurales entre predictores, y definir las matrices parsimoniosas de entrada para Componentes Principales (PCA) y Mínimos Cuadrados Parciales (PLS) considerando el tamaño muestral ($n=20$).

In [ ]:
# 19.1 Diagnóstico de Colinealidad y Redundancia (|rho| >= 0.65)
report_collin = bt.collinearity_summary(threshold=0.65)
display(report_collin)

# 20.1 Preparación de Bloques para PCA
report_pca = bt.pca_candidates(metal="Plomo_ug_dL")
display(report_pca)

# 21.1 Definición de Matrices X e Y para PLS
report_pls = bt.pls_candidates(metal="Plomo_ug_dL")
display(report_pls)

--- 
## Exportación de Resultados

Todos los reportes generados pueden exportarse a libros de Excel con múltiples hojas, archivos CSV y páginas HTML Booktabs completas.

In [ ]:
# Exportar reporte maestro a Excel y HTML
output_dir = Path("output_bivariante")
output_dir.mkdir(parents=True, exist_ok=True)

report_master_pb.to_excel(str(output_dir / "matriz_maestra_plomo.xlsx"))
report_master_pb.to_html(str(output_dir / "matriz_maestra_plomo.html"), full_page=True)
report_cat_pb.to_excel(str(output_dir / "analisis_sectores_kruskal_dunn.xlsx"))

print(f"Reportes exportados exitosamente en: {output_dir.resolve()}")